### Read file from volume and load into Raw table
- Set to Current Schema.
- Read files from Volume - Write into Table
- Validate the table


In [0]:
%sql
use schema investment_vision;
select current_catalog(), current_schema();

In [0]:
raw_df = spark.read.format('csv').options(
    header=True,
    inferSchema=True,
    multiLine=True,
    quote='"'
).load(
    '/Volumes/workspace/investment_vision/files_volume/Invest_Dec.csv',
    sep=','
)

cleaned_columns = [
    col.replace(' ', '_').replace(',', '').replace(';', '').replace('{', '').replace('}', '')
    .replace('(', '').replace(')', '').replace('\n', '').replace('\t', '').replace('=', '')
    for col in raw_df.columns
]

raw_df = raw_df.toDF(*cleaned_columns)

display(raw_df)

raw_df.write.mode('append').option('mergeSchema', 'true').saveAsTable(
    'workspace.investment_vision.raw_till_nov')

# Create a temporary view for raw_df
raw_df.createOrReplaceTempView('raw_till_nov_temp')

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.investment_vision.history_table (
    -- Activity_Date DATE,
    -- Process_Date DATE,
    Settle_Date DATE,
    Settle_Month_Year STRING,
    Instrument STRING,
    -- Description STRING,
    Trans_Code STRING,
    Quantity String,
    Price STRING,
    Amount STRING
)
USING DELTA

In [0]:
%sql
MERGE INTO investment_vision.history_table AS target
USING (
    SELECT
        to_date(rsa.date, 'MM/dd/yyyy') AS Settle_Date,
        date_format(to_date(rsa.date, 'MM/dd/yyyy'), 'MMM-yyyy') AS Settle_Month_Year,
        rsa.ticker AS Instrument,
        rsa.transaction_type AS Trans_Code,
        rsa.qty AS Quantity,
        rsa.current_price AS Price,
        rsa.amount AS Amount
    FROM investment_vision.raw_statement_activity rsa
    WHERE to_date(rsa.date, 'MM/dd/yyyy') IS NOT NULL
    UNION ALL
    SELECT
        ri.Settle_Date AS Settle_Date,
        date_format(ri.Settle_Date, 'MMM-yyyy') AS Settle_Month_Year,
        ri.Instrument AS Instrument,
        ri.Trans_Code AS Trans_Code,
        ri.Quantity AS Quantity,
        ri.Price AS Price,
        ri.Amount AS Amount
    FROM raw_till_nov_temp ri
    WHERE ri.Settle_Date IS NOT NULL
) AS source
ON 
   target.Quantity = source.Quantity
   AND target.Settle_Date = source.Settle_Date
WHEN NOT MATCHED THEN
    INSERT (
        Instrument, Trans_Code, Quantity, Price, Amount, Settle_Date, Settle_Month_Year
    ) VALUES (
        source.Instrument, source.Trans_Code, source.Quantity, source.Price, source.Amount, source.Settle_Date, source.Settle_Month_Year
    )

In [0]:
%sql
--drop table investment_vision.history_table
select * from investment_vision.history_table